# 03 Prepare Power BI Tables

Prepare the final PostgreSQL/Power BI-ready mart tables for the DDM submission.

The session is the central unit of analysis. `userId` is sparse and not reliable enough for user-level RFM, CLV, churn, or segmentation to be the main project story. CTR and revenue fields are offline proxies only.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs/project_config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

from ddm.io import load_config
from ddm.pipeline import prepare_powerbi_marts


## Build Mart Tables

In [ ]:
config = load_config(PROJECT_ROOT / "configs/project_config.yaml")
outputs = prepare_powerbi_marts(PROJECT_ROOT)
outputs

## Required Tables

In [ ]:
mart_root = PROJECT_ROOT / config["outputs"]["mart_root"]
required_tables = [
    "dim_item",
    "dim_model",
    "fact_session_summary",
    "fact_purchases",
    "fact_test_examples",
    "fact_recommendations",
    "fact_recommendation_eval",
    "fact_metrics",
    "fact_marketing_kpis",
]
summary_rows = []
for table in required_tables:
    path = mart_root / f"{table}.parquet"
    df = pd.read_parquet(path)
    summary_rows.append({"table": table, "path": str(path.relative_to(PROJECT_ROOT)), "rows": len(df), "columns": df.shape[1]})
pd.DataFrame(summary_rows)


## Table Previews

In [ ]:
for table in required_tables:
    path = mart_root / f"{table}.parquet"
    df = pd.read_parquet(path)
    print(f"\n{table}: {df.shape}")
    display(df.head())

## Power BI Notes

- Session is the central unit of analysis.
- `userId` is not reliable enough for user-level RFM, CLV, churn, or segmentation as the main analysis.
- CTR is proxy only: `CTR Proxy@20` equals offline HR@20, not real CTR.
- Revenue is proxy only: GMV/value metrics use `pricelog2`-derived `price_proxy`.
- Relative-delta values are offline benchmark deltas, not causal marketing impact.

In [ ]:
notes_path = mart_root / "powerbi_notes.md"
print(notes_path.read_text())